<div style="background-color: black; color: white; padding: 10px;text-align: center;">
  <strong>Date Published:</strong> Nov 20, 2026 <strong>Author:</strong> Adnan Alaref
</div>

# Assignment Problem

The **Assignment Problem** is an optimization problem that aims to find the optimal assignment of tasks to agents while minimizing the total assignment cost.

Each agent is assigned to exactly one task, and each task is assigned to exactly one agent.

### Hungarian Algorithm

The **Hungarian Algorithm** is a combinatorial optimization algorithm used to solve the Assignment Problem optimally.

Its main steps are:

1. **Row Reduction:** Subtract the minimum value of each row from all elements in that row.
2. **Column Reduction:** Subtract the minimum value of each column from all elements in that column.
3. **Independent Zeros:** Find the maximum number of independent zeros using Bipartite Matching.
4. **Minimum Line Cover:** Cover all zeros using the minimum number of horizontal and vertical lines.
5. **Matrix Adjustment:** If the number of covering lines is less than the matrix size, adjust the matrix to generate additional zeros.
6. **Optimal Assignment:** Repeat the process until a complete assignment is found, then calculate the minimum total cost using the original cost matrix.

### Implementation Approaches

The algorithm can be implemented in two ways:

- **From Scratch:** Implementing each step of the Hungarian Algorithm using NumPy.
- **Using SciPy:** Applying SciPy's built-in implementation through `scipy.optimize.linear_sum_assignment`.

Both approaches find an optimal assignment that minimizes the total cost.

In [1]:
import numpy as np

import warnings
warnings.simplefilter(action="ignore")
warnings.filterwarnings(action="ignore", category=FutureWarning)

# **Hungarian Algorithm — Using SciPy's Built-in Implementation**

In [2]:
from scipy.optimize import linear_sum_assignment

cost_matrix = np.array([
    [4, 1, 3],
    [2, 0, 5],
    [3, 2, 2]
])

row_ind, col_ind = linear_sum_assignment(cost_matrix)

assignment = list(zip(row_ind, col_ind))
total_cost = cost_matrix[row_ind, col_ind].sum()

print("Assignment:", assignment)
print("Total Cost:", total_cost)

Assignment: [(np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(2), np.int64(2))]
Total Cost: 5


In [3]:
from scipy.optimize import linear_sum_assignment

cost_matrix = np.array([
    [40, 60, 15],
    [25, 30, 45],
])

row_ind, col_ind = linear_sum_assignment(cost_matrix)

assignment = list(zip(row_ind, col_ind))
total_cost = cost_matrix[row_ind, col_ind].sum()

print("Assignment:", assignment)
print("Total Cost:", total_cost)

Assignment: [(np.int64(0), np.int64(2)), (np.int64(1), np.int64(0))]
Total Cost: 40


# **Hungarian Algorithm — From Scratch**

In [4]:
# 1. ───── Row Reduction ─────

def row_reduction(matrix: np.ndarray) -> np.ndarray:
    """
    Subtract the minimum value of each row
    from all elements in that row.
    """
    row_min = matrix.min(axis=1, keepdims=True)
    return matrix - row_min


# 2. ───── Column Reduction ─────

def column_reduction(matrix: np.ndarray) -> np.ndarray:
    """
    Subtract the minimum value of each column
    from all elements in that column.
    """
    col_min = matrix.min(axis=0, keepdims=True)
    return matrix - col_min


# 3. ───── Find Independent Zeros ─────

def find_independent_zeros(matrix: np.ndarray) -> tuple[list[tuple[int, int]], list[int]]:
    """
    Find the maximum number of independent zeros
    using Maximum Bipartite Matching.
    """
    n = matrix.shape[0]

    zeros = [
        np.where(matrix[i] == 0)[0].tolist()
        for i in range(n)
    ]

    col_match = [-1] * n

    def dfs(row: int, visited: list[bool]) -> bool:
        for col in zeros[row]:
            if visited[col]:
                continue

            visited[col] = True

            if col_match[col] == -1 or dfs(col_match[col], visited):
              col_match[col] = row
              return True

        return False

    # Try to assign a column to every row.
    for row in range(n):
      dfs(row, [False] * n)

    # Convert column assignments into (row, column) pairs.
    independent_zeros = [
      (row, col)
      for col, row in enumerate(col_match)
      if row != -1
    ]

    return independent_zeros, col_match


# 4. ───── Minimum Line Cover ─────

def minimum_line_cover(matrix: np.ndarray, col_match: list[int]) -> tuple[list[int], list[int]]:
    """
    Find the minimum number of horizontal and vertical
    lines needed to cover all zeros.
    """
    n = len(col_match)

    zeros = [
        np.where(matrix[i] == 0)[0].tolist()
        for i in range(n)
    ]

    visited_rows = [False] * n
    visited_cols = [False] * n

    # Find unmatched rows.
    for row in range(n):
        if row not in col_match:
            visited_rows[row] = True

    def dfs(row: int) -> None:
        for col in zeros[row]:

            if col_match[col] == row or visited_cols[col]:
                continue

            visited_cols[col] = True

            matched_row = col_match[col]

            if matched_row != -1 and not visited_rows[matched_row]:
                visited_rows[matched_row] = True
                dfs(matched_row)

    # Start DFS from unmatched rows.
    for row in range(n):
        if visited_rows[row]:
            dfs(row)

    # Minimum line cover.
    cover_rows = [
        row for row in range(n)
        if not visited_rows[row]
    ]

    cover_cols = [
        col for col in range(n)
        if visited_cols[col]
    ]

    return cover_rows, cover_cols


# 5. ───── Matrix Adjustment ─────

def matrix_adjustment(matrix: np.ndarray, cover_rows: list[int], cover_cols: list[int]) -> np.ndarray:
    """
    Create additional zeros by adjusting the matrix.
    """
    matrix = matrix.copy()

    n = matrix.shape[0]

    row_covered = np.zeros(n, dtype=bool)
    col_covered = np.zeros(n, dtype=bool)

    row_covered[cover_rows] = True
    col_covered[cover_cols] = True

    # Identify uncovered elements.
    uncovered = (
        ~row_covered[:, np.newaxis]
        & ~col_covered[np.newaxis, :]
    )

    # Find minimum uncovered value.
    min_uncovered = matrix[uncovered].min()

    # Subtract from uncovered elements.
    matrix[uncovered] -= min_uncovered

    # Add to intersections of two covering lines.
    matrix[np.ix_(cover_rows, cover_cols)] += min_uncovered

    return matrix


# 6. ───── Hungarian Algorithm ─────

def hungarian_algorithm(cost_matrix: np.ndarray) -> tuple[list[tuple[int, int]], float]:
    """
    Solve the square Assignment Problem using
    the Hungarian Algorithm.
    """
    if cost_matrix.ndim != 2:
        raise ValueError("Input must be a 2D matrix.")

    if cost_matrix.shape[0] != cost_matrix.shape[1]:
        raise ValueError("Matrix must be square.")

    if cost_matrix.shape[0] == 0:
        raise ValueError("Matrix must not be empty.")

    if not np.isfinite(cost_matrix).all():
        raise ValueError("Matrix must contain finite values.")

    original_matrix = cost_matrix.astype(float).copy()

    # Step 1: Row Reduction.
    reduced_matrix = row_reduction(original_matrix)

    # Step 2: Column Reduction.
    reduced_matrix = column_reduction(reduced_matrix)

    while True:

        # Step 3: Find independent zeros.
        independent_zeros, col_match = find_independent_zeros(
          reduced_matrix
        )

        # Check for complete assignment.
        if len(independent_zeros) == len(original_matrix):
            break

        # Step 4: Find minimum line cover.
        cover_rows, cover_cols = minimum_line_cover(
            reduced_matrix,
            col_match
        )

        # Step 5: Adjust the matrix.
        reduced_matrix = matrix_adjustment(
            reduced_matrix,
            cover_rows,
            cover_cols
        )

    # Final assignment.
    assignment = independent_zeros

    # Calculate total cost from original matrix.
    total_cost = sum(
        original_matrix[row, col]
        for row, col in assignment
    )

    return assignment, float(total_cost)

# << ─────  Example  ───── >>

In [5]:
cost_matrix = np.array([
    [4, 1, 3],
    [2, 0, 5],
    [3, 2, 2]
])

assignment, total_cost = hungarian_algorithm(cost_matrix)

print("Assignment:", assignment)
print("Total Cost:", total_cost)

Assignment: [(1, 0), (0, 1), (2, 2)]
Total Cost: 5.0


In [6]:
cost_matrix = np.array([
    [9, 2, 7],
    [6, 4, 3],
    [5, 8, 1]
])

assignment, total_cost = hungarian_algorithm(cost_matrix)

print("Assignment:", assignment)
print("Total Cost:", total_cost)

Assignment: [(1, 0), (0, 1), (2, 2)]
Total Cost: 9.0


In [7]:
cost_matrix = np.array([
    [40, 60, 15],
    [25, 30, 45],
    [55, 30, 25]
])

assignment, total_cost = hungarian_algorithm(cost_matrix)

print("Assignment:", assignment)
print("Total Cost:", total_cost)

Assignment: [(1, 0), (2, 1), (0, 2)]
Total Cost: 70.0


In [8]:
cost_matrix = np.array([
    [30, 25, 10],
    [15, 10, 20],
    [25, 20, 15]
])

assignment, total_cost = hungarian_algorithm(cost_matrix)

print("Assignment:", assignment)
print("Total Cost:", total_cost)

Assignment: [(2, 0), (1, 1), (0, 2)]
Total Cost: 45.0


# <a id="Import"></a><div style="background: linear-gradient(to right, #1b5e20, #2e7d32, #388e3c, #43a047, #4caf50); font-family: 'Times New Roman', serif; font-size: 28px; font-weight: bold; text-align: center; border-radius: 15px; padding: 15px; border: 2px solid #ffffff; box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Thanks & Upvote ❤️</div>